## Importation des bibliothèques et des données

In [25]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import io
import requests
import numpy as np
import json
from shapely.geometry import shape
np.float = float
np.long = int
import sys
sys.modules["numba"] = None

url_ips = "https://data.education.gouv.fr/explore/dataset/fr-en-ips_lycees/download/?format=csv&timezone=Europe/Berlin&lang=fr&use_labels_for_header=true&csv_separator=%3B"
url_res = "https://data.education.gouv.fr/explore/dataset/fr-en-indicateurs-de-resultat-des-lycees-gt_v2/download/?format=csv&timezone=Europe/Berlin&lang=fr&use_labels_for_header=true&csv_separator=%3B"
url_carte = "https://data-interne.ademe.fr/data-fair/api/v1/datasets/geo-contours-departements/lines?draft=false&size=10000&finalizedAt=2022-03-02T14:11:30.661Z&page=1&format=csv&sep=;"

df_ips = pd.read_csv(io.StringIO(requests.get(url_ips).text), sep=";")
df_res = pd.read_csv(io.StringIO(requests.get(url_res).text), sep=";")
df_carte = pd.read_csv(url_carte, sep=";") 

df_ips.head(5)

,Rentrée scolaire,Académie,Code du département,Département,UAI,Nom de l'établissment,Code INSEE de la commune,Nom de la commune,Secteur,Type de lycée,IPS voie GT,IPS voie PRO,IPS Ensemble GT-PRO,Ecart-type de l'IPS voie GT,Ecart-type de l'IPS voie PRO
0,2018-2019,DIJON,58.0,NIEVRE,0580031U,LYCEE GENERAL ET TECHNOLOGIQUE JULES RENARD,58194.0,NEVERS,public,LEGT,120.1,NaN,120.1,NaN,NaN
1,2018-2019,DIJON,58.0,NIEVRE,0580042F,LYCEE PROFESSIONNEL LE MONT CHATELET,58304.0,VARZY,public,LP,NaN,97.1,97.1,NaN,NaN
2,2018-2019,DIJON,58.0,NIEVRE,0580076T,LYCEE TECHNOLOGIQUE ET PROFESSIONNEL PRIVE SIM...,58086.0,COSNE COURS SUR LOIRE,privé sous contrat,LPO,94.7,89.8,92.5,NaN,NaN
3,2018-2019,LILLE,59.0,NORD,0590010R,LYCEE GENERAL ET TECHNOLOGIQUE PAUL HAZARD,59017.0,ARMENTIERES,public,LEGT,103.3,NaN,103.3,NaN,NaN
4,2018-2019,LILLE,59.0,NORD,0590037V,LYCEE PROFESSIONNEL LOUISE DE BETTIGNIES,59122.0,CAMBRAI,public,LP,NaN,73.4,73.4,NaN,NaN


In [7]:
df_res.head(5)

,Année,UAI,Etablissement,Secteur,Code commune,Commune,Code departement,Département,Académie,Code région,...,Valeur ajoutée du taux de mentions - S2TMD,Valeur ajoutée du taux de mentions - STHR,Nombre de mentions TB avec félicitations - G,Nombre de mentions TB sans félicitations - G,Nombre de mentions B - G,Nombre de mentions AB - G,Nombre de mentions TB avec félicitations - T,Nombre de mentions TB sans félicitations - T,Nombre de mentions B - T,Nombre de mentions AB - T
0,2017,0290108K,LYCEE DUPUY DE LOME (GENERAL ET TECHNO.),public,29019,BREST,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017,0290156M,LYCEE ST LOUIS,privé sous contrat,29026,CHATEAULIN,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017,0290174G,LYCEE NOTRE DAME DE KERBERTRA,privé sous contrat,29233,QUIMPERLE,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017,0290177K,LYCEE DU KREISKER,privé sous contrat,29259,ST POL DE LEON,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017,0290181P,LYCEE JAVOUHEY,privé sous contrat,29019,BREST,29,FINISTERE,RENNES,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Ici on a importé les deux tableaux dont on a besoin avec l'IPS (indice de position sociale) et les résultats à l'examen du baccalauréat. On ne les a pas importé directement avec pd.read_csv car cela bloquait parfois selon l'envirronement python du terminal utilisé. Avec la bibliothèque requests adaptée aux API au moins on est sur de ne pas avoir ce problème de versions.
De plus on a importé un shapefile avec les délimitations administratives des départements français pour pouvoir faire des cartes plus tard.

## Nettoyage et mise en forme des données

In [18]:
df_ips[df_ips["Code du département"].isna()].head(5)

,Rentrée scolaire,Académie,Code du département,Département,UAI,Nom de l'établissment,Code INSEE de la commune,Nom de la commune,Secteur,Type de lycée,IPS voie GT,IPS voie PRO,IPS Ensemble GT-PRO,Ecart-type de l'IPS voie GT,Ecart-type de l'IPS voie PRO
602,2019-2020,CORSE,NaN,CORSE-DU-SUD,6200004K,LYCEE PROFESSIONNEL FINOSELLO AJACCIO,NaN,AJACCIO,public,LP,NaN,86.2,86.2,NaN,27.3
603,2019-2020,CORSE,NaN,HAUTE-CORSE,7200021K,LYCEE GENERAL ET TECHNOLOGIQUE PASCAL PAOLI CORTE,NaN,CORTE,public,LEGT,111.8,NaN,111.8,31.4,NaN
1845,2016-2017,CORSE,NaN,CORSE-DU-SUD,6200001G,LYCEE GENERAL ET TECHNOLOGIQUE FESCH AJACCIO,NaN,AJACCIO,public,LEGT,113.2,NaN,113.2,NaN,NaN
1846,2016-2017,CORSE,NaN,CORSE-DU-SUD,6200002H,LYCEE GENERAL ET TECHNOLOGIQUE LAETITIA BONAPA...,NaN,AJACCIO,public,LEGT,112.5,NaN,112.5,NaN,NaN
1847,2016-2017,CORSE,NaN,CORSE-DU-SUD,6200004K,LYCEE PROFESSIONNEL FINOSELLO AJACCIO,NaN,AJACCIO,public,LP,NaN,87.6,87.6,NaN,NaN


In [10]:
df_res[df_res["Code departement"].isna()].head(5)

,Année,UAI,Etablissement,Secteur,Code commune,Commune,Code departement,Département,Académie,Code région,...,Valeur ajoutée du taux de mentions - S2TMD,Valeur ajoutée du taux de mentions - STHR,Nombre de mentions TB avec félicitations - G,Nombre de mentions TB sans félicitations - G,Nombre de mentions B - G,Nombre de mentions AB - G,Nombre de mentions TB avec félicitations - T,Nombre de mentions TB sans félicitations - T,Nombre de mentions B - T,Nombre de mentions AB - T


Ici on voit que les résultats au bac des lycées corses sont manquants donc on va enlever ces lignes dans le tableau sur l'IPS. 
Ensuite on va créer une colonne correspondant à l'année dans df_ips pour pouvoir merge les deux tableaux ensemble sur le lycée et l'année scolaire.

In [26]:
df_ips = df_ips[df_ips["Code du département"].notna()]
df_ips["Année"] = df_ips["Rentrée scolaire"].str[-4:].astype(int)
df_ips["Code du département"] = pd.to_numeric(df_ips["Code du département"], errors="coerce").astype("Int64")
df_res["Code departement"] = pd.to_numeric(df_res["Code departement"], errors="coerce").astype("Int64")
df = df_ips.merge(df_res, on=["UAI","Année"], how="left", suffixes=("_ips","_res"))
df.head(5) 

,Rentrée scolaire,Académie_ips,Code du département,Département_ips,UAI,Nom de l'établissment,Code INSEE de la commune,Nom de la commune,Secteur_ips,Type de lycée,...,Valeur ajoutée du taux de mentions - S2TMD,Valeur ajoutée du taux de mentions - STHR,Nombre de mentions TB avec félicitations - G,Nombre de mentions TB sans félicitations - G,Nombre de mentions B - G,Nombre de mentions AB - G,Nombre de mentions TB avec félicitations - T,Nombre de mentions TB sans félicitations - T,Nombre de mentions B - T,Nombre de mentions AB - T
0,2018-2019,DIJON,58,NIEVRE,0580031U,LYCEE GENERAL ET TECHNOLOGIQUE JULES RENARD,58194.0,NEVERS,public,LEGT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-2019,DIJON,58,NIEVRE,0580042F,LYCEE PROFESSIONNEL LE MONT CHATELET,58304.0,VARZY,public,LP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-2019,DIJON,58,NIEVRE,0580076T,LYCEE TECHNOLOGIQUE ET PROFESSIONNEL PRIVE SIM...,58086.0,COSNE COURS SUR LOIRE,privé sous contrat,LPO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-2019,LILLE,59,NORD,0590010R,LYCEE GENERAL ET TECHNOLOGIQUE PAUL HAZARD,59017.0,ARMENTIERES,public,LEGT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-2019,LILLE,59,NORD,0590037V,LYCEE PROFESSIONNEL LOUISE DE BETTIGNIES,59122.0,CAMBRAI,public,LP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Enfin on met juste en forme le tableau avec les données géographiques.

In [27]:
df_carte["geometry"] = df_carte["geometry"].apply(
    lambda x: shape(json.loads(x)) if isinstance(x, str) else x)
df_carte["DDEP_C_COD"] = pd.to_numeric(df_carte["DDEP_C_COD"], errors="coerce").astype("Int64")
df_carte = df_carte.rename(columns={"DDEP_L_LIB":"Département","DDEP_C_COD":"Code du département", "DREG_L_LIB":"Région"})
df_carte = gpd.GeoDataFrame(df_carte, geometry="geometry")